In [1]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.append(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from utils.seed import set_seed

set_seed(42, verbose=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

[Seed] Random seed set to 42
Device: cuda


In [4]:
#calculates knowledge distillation loss
def vanilla_kd_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    targets: torch.Tensor,
    temperature: float = 4.0,
    alpha: float = 0.9,
):
    # Hard-label supervision (student vs ground truth)
    hard_loss = F.cross_entropy(student_logits, targets)

    # Soft-label distillation (student mimics teacher's softened distribution)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1) #student soft probs
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    soft_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    ) * (temperature ** 2)

    # Weighted blend of both signals
    total_loss = alpha * soft_loss + (1.0 - alpha) * hard_loss

    return total_loss, hard_loss.detach(), soft_loss.detach()

In [5]:
# Sanity check 
torch.manual_seed(0)

batch_size, num_classes = 4, 10
fake_student = torch.randn(batch_size, num_classes)
fake_teacher = torch.randn(batch_size, num_classes)
fake_targets = torch.randint(0, num_classes, (batch_size,))

total, hard, soft = vanilla_kd_loss(
    fake_student, fake_teacher, fake_targets,
    temperature=4.0, alpha=0.9,
)

print(f"Total loss: {total.item():.4f}")
print(f"Hard  loss: {hard.item():.4f}")
print(f"Soft  loss: {soft.item():.4f}")

# Sanity property: if student == teacher, soft loss should be ~0
_, _, soft_same = vanilla_kd_loss(
    fake_teacher, fake_teacher, fake_targets,
    temperature=4.0, alpha=0.9,
)
print(f"Soft loss when student==teacher (expect ~0): {soft_same.item():.6f}")

Total loss: 1.0906
Hard  loss: 2.6152
Soft  loss: 0.9212
Soft loss when student==teacher (expect ~0): 0.000000


In [6]:
import torch.nn as nn
from torchvision.models import resnet34, resnet18


def adapt_resnet_for_cifar(model: nn.Module, num_classes: int = 10) -> nn.Module:
    # Replace ImageNet 7x7 stride-2 stem with a CIFAR-friendly 3x3 stride-1 stem
    model.conv1 = nn.Conv2d(
        in_channels=3, out_channels=64,
        kernel_size=3, stride=1, padding=1, bias=False,
    )
    # Remove the initial maxpool (too aggressive for 32x32 inputs)
    model.maxpool = nn.Identity()
    # Resize the final classifier to our number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_teacher(num_classes: int = 10) -> nn.Module:
    model = resnet34(weights=None)          # random init (we train from scratch)
    return adapt_resnet_for_cifar(model, num_classes)


# Quick check: build and inspect output shape
teacher = build_teacher(num_classes=10).to(device)
dummy = torch.randn(2, 3, 32, 32).to(device)
out = teacher(dummy)
print("Teacher output shape:", out.shape)  

Teacher output shape: torch.Size([2, 10])


In [7]:
def build_student(num_classes: int = 10) -> nn.Module:
    model = resnet18(weights=None)          # from scratch, same as teacher
    return adapt_resnet_for_cifar(model, num_classes)


# Quick check: build and inspect output shape + parameter counts
student = build_student(num_classes=10).to(device)
out = student(dummy)
print("Student output shape:", out.shape)   # expect [2, 10]

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

teacher_params = count_params(teacher)
student_params = count_params(student)
print(f"Teacher params: {teacher_params:,}")
print(f"Student params: {student_params:,}")
print(f"Compression ratio (teacher/student): {teacher_params / student_params:.2f}x")

Student output shape: torch.Size([2, 10])
Teacher params: 21,282,122
Student params: 11,173,962
Compression ratio (teacher/student): 1.90x


In [8]:
import torchvision.transforms as T

# Precomputed CIFAR-10 channel statistics (standard values from literature)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Training transform: light augmentation + normalization
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Eval transform: NO augmentation, only normalization
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

print("Transforms ready.")

Transforms ready.


In [ ]:
#ONLY ONCE

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10

DATA_ROOT = "/content/drive/MyDrive/kd_experiments/data" if os.path.exists("/content") else "../data"
SAVE_DIR  = "/content/drive/MyDrive/kd_experiments" if os.path.exists("/content") else "../results"
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 128
VAL_SIZE = 5000          # carved out of the 50k training images
NUM_WORKERS = 2

# Load the training pool twice with different transforms, same underlying images
train_pool_aug = CIFAR10(DATA_ROOT, train=True, download=True, transform=train_transform)
train_pool_eval = CIFAR10(DATA_ROOT, train=True, download=True, transform=eval_transform)

# Official test set (never touched during training/tuning)
test_set = CIFAR10(DATA_ROOT, train=False, download=True, transform=eval_transform)

# Reproducible split of indices 0..49999 into train / val
num_train_total = len(train_pool_aug)                       # 50000
generator = torch.Generator().manual_seed(42)               # fixed split, every run
perm = torch.randperm(num_train_total, generator=generator).tolist()
val_indices = perm[:VAL_SIZE]
train_indices = perm[VAL_SIZE:]

# Train uses augmented pool; val uses eval pool (same images, no augmentation)
train_set = Subset(train_pool_aug, train_indices)
val_set = Subset(train_pool_eval, val_indices)

# DataLoaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, drop_last=False)

print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")

100.0%


Train: 45000  Val: 5000  Test: 10000


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score

@torch.no_grad()
def evaluate(model: nn.Module, loader, device) -> dict:
    model.eval()

    all_preds = []
    all_targets = []
    top1_correct = 0
    top5_correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)

        # Top-1
        top1_preds = logits.argmax(dim=1)
        top1_correct += (top1_preds == labels).sum().item()

        # Top-5
        top5_preds = logits.topk(5, dim=1).indices
        top5_correct += sum(labels[i].item() in top5_preds[i].tolist()
                            for i in range(labels.size(0)))

        total += labels.size(0)
        all_preds.extend(top1_preds.cpu().tolist())
        all_targets.extend(labels.cpu().tolist())

    top1_acc = 100.0 * top1_correct / total
    top5_acc = 100.0 * top5_correct / total
    precision = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_targets, all_preds, average="macro", zero_division=0)
    f1        = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    return {
        "top1_acc":  round(top1_acc, 4),
        "top5_acc":  round(top5_acc, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [12]:
import yaml

with open("../configs/cifar10.yaml") as f:
    cfg = yaml.safe_load(f)
with open("../configs/kd_config.yaml") as f:
    kd_cfg = yaml.safe_load(f)

EPOCHS       = cfg["training"]["epochs"]
LR           = cfg["training"]["learning_rate"]
MOMENTUM     = cfg["training"]["momentum"]
WEIGHT_DECAY = cfg["training"]["weight_decay"]
T_MAX        = cfg["training"]["t_max"]
BATCH_SIZE   = cfg["dataloader"]["batch_size"]
VAL_SIZE     = cfg["dataloader"]["val_size"]

TEMPERATURE  = kd_cfg["vanilla_kd"]["temperature"]
ALPHA        = kd_cfg["vanilla_kd"]["alpha"]

print(f"Epochs: {EPOCHS}, LR: {LR}, Momentum: {MOMENTUM}, WD: {WEIGHT_DECAY}")
print(f"KD Temperature: {TEMPERATURE}, Alpha: {ALPHA}")

Epochs: 200, LR: 0.1, Momentum: 0.9, WD: 0.0005
KD Temperature: 4.0, Alpha: 0.9


In [13]:
import torch.optim as optim
import json

def train_teacher(model, train_loader, val_loader, epochs, device, save_path):
    optimizer = optim.SGD(
        model.parameters(),
        lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

    best_val_acc = 0.0
    history = {"train_loss": [], "val_top1": []}

    for epoch in range(1, epochs + 1):
        # ── Training phase ──────────────────────────────
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        # ── Validation phase ─────────────────────────────
        val_metrics = evaluate(model, val_loader, device)
        val_acc = val_metrics["top1_acc"]

        history["train_loss"].append(avg_loss)
        history["val_top1"].append(val_acc)

        # ── Checkpoint: save best model only ─────────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Val Top-1: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")

    print(f"\nTeacher training done. Best val acc: {best_val_acc:.2f}%")
    return history

In [ ]:
#Colab training
teacher = build_teacher(num_classes=10).to(device)
TEACHER_SAVE_PATH = f"{SAVE_DIR}/teacher.pth"

teacher_history = train_teacher(
    model=teacher,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    device=device,
    save_path=TEACHER_SAVE_PATH,
)

# Save metrics to Drive
with open(f"{SAVE_DIR}/teacher_history.json", "w") as f:
    json.dump(teacher_history, f)

print(f"Teacher saved to: {TEACHER_SAVE_PATH}")

In [ ]:
def train_baseline(model, train_loader, val_loader, epochs, device, save_path):
    optimizer = optim.SGD(
        model.parameters(),
        lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

    best_val_acc = 0.0
    history = {"train_loss": [], "val_top1": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        val_metrics = evaluate(model, val_loader, device)
        val_acc = val_metrics["top1_acc"]

        history["train_loss"].append(avg_loss)
        history["val_top1"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Val Top-1: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")

    print(f"\nBaseline training done. Best val acc: {best_val_acc:.2f}%")
    return history

In [ ]:
baseline_student = build_student(num_classes=10).to(device)
BASELINE_SAVE_PATH = f"{SAVE_DIR}/baseline_student.pth"

baseline_history = train_baseline(
    model=baseline_student,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    device=device,
    save_path=BASELINE_SAVE_PATH,
)

with open(f"{SAVE_DIR}/baseline_history.json", "w") as f:
    json.dump(baseline_history, f)

print(f"Baseline student saved to: {BASELINE_SAVE_PATH}")

In [ ]:
def train_vanilla_kd(student, teacher, train_loader, val_loader, epochs, device, save_path):
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False

    optimizer = optim.SGD(
        student.parameters(),
        lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

    best_val_acc = 0.0
    history = {"train_loss": [], "hard_loss": [], "soft_loss": [], "val_top1": []}

    for epoch in range(1, epochs + 1):
        student.train()
        running_total = running_hard = running_soft = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacher(images)

            optimizer.zero_grad()
            student_logits = student(images)

            total_loss, hard_loss, soft_loss = vanilla_kd_loss(
                student_logits, teacher_logits, labels,
                temperature=TEMPERATURE, alpha=ALPHA,
            )
            total_loss.backward()
            optimizer.step()

            running_total += total_loss.item()
            running_hard  += hard_loss.item()
            running_soft  += soft_loss.item()

        scheduler.step()
        n = len(train_loader)
        val_metrics = evaluate(student, val_loader, device)
        val_acc = val_metrics["top1_acc"]

        history["train_loss"].append(running_total / n)
        history["hard_loss"].append(running_hard / n)
        history["soft_loss"].append(running_soft / n)
        history["val_top1"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(student.state_dict(), save_path)

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {running_total/n:.4f} | Hard: {running_hard/n:.4f} | Soft: {running_soft/n:.4f} | Val: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")

    print(f"\nVanilla KD training done. Best val acc: {best_val_acc:.2f}%")
    return history

In [ ]:
# Load best teacher weights before distillation
teacher.load_state_dict(torch.load(TEACHER_SAVE_PATH, map_location=device))

vanilla_student = build_student(num_classes=10).to(device)
VANILLA_SAVE_PATH = f"{SAVE_DIR}/vanilla_student.pth"

vanilla_history = train_vanilla_kd(
    student=vanilla_student,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    device=device,
    save_path=VANILLA_SAVE_PATH,
)

with open(f"{SAVE_DIR}/vanilla_history.json", "w") as f:
    json.dump(vanilla_history, f)

print(f"Vanilla KD student saved to: {VANILLA_SAVE_PATH}")